# `04_bicycle_route_m_ways_distinct_classified`: Tag resolution pipeline for bicycle route ways

## Introduction

### Purpose

This notebook derives `bicycle_route_m_ways_distinct_classified`, the fully resolved version of `bicycle_route_m_ways_distinct`. It takes every way in the designated bicycle route network and works through its OSM tags to determine, for each direction of travel, what bicycle access applies, which surface a cyclist uses, and how confidently cycling infrastructure can be identified. The grain is one row per way throughout: 181,318 ways in, 181,318 ways out. Per the thesis (§3.4.1), this is the stage-04 tag-resolution work that prepares the per-way table for the per-side classification step that follows.

### Inputs

- `bicycle_route_m_ways_distinct` from `00_bicycle_route_relations` (181,318 ways totalling 38,570.854 km, clipped to the Netherlands administrative boundary).

### Outputs

- `bicycle_route_m_ways_distinct_classified`: one row per way, enriched with mapping style, per-side infrastructure labels, way-level infrastructure label, forward and backward bicycle access signals (with explicitness score), forward and backward routing surfaces, road and bicycle directionality, and the final `infrastructure_confidence` column. Geometry and length columns are carried through unchanged.

### Key steps

The pipeline proceeds in nine column-additive steps. Step 1 extracts raw tag values into named columns. Step 2 resolves conflicting `cycleway:*` tags into single effective values per side using the deterministic precedence `cycleway:right` / `cycleway:left` > `cycleway:both` > `cycleway` (thesis §3.4.1). Step 3 assigns a `mapping_style` (`dedicated_mapping` / `carriageway_mapping` / `n/a` / `tagging_conflict` / `ferry`). Step 4 classifies each carriageway side independently (`cycling_infrastructure` / `no_cycling_infrastructure` / `separate_infrastructure` / `NULL` / `n/a`); the `NULL`-vs-`no` distinction is the basis for confidence scoring downstream and underpins the inferred-absence categories the thesis builds in §3.4.3 and §3.4.4. Step 5 combines the two sides into a single way-level label. Step 6 computes a forward and backward bicycle-access signal with an explicitness score (1.00 explicit bicycle, 0.95 vehicle, 0.90 access, 0.75 inferred from infrastructure, 0.20–0.85 from highway-type defaults). Step 7 resolves the routing surface per direction. Step 8 reads `oneway=*` to set road directionality. Step 9 closes the pipeline by applying cycling-specific overrides for bicycle directionality and producing `infrastructure_confidence`.

### Dependencies on prior notebooks

- `00_bicycle_route_relations.ipynb`: provides `bicycle_route_m_ways_distinct`.
- `03_boundaries_population.ipynb`: brought in via `%run` for the boundary and population variables it exposes (and the transitive `%run` of `00_bicycle_route_relations` and `01_boundaries`).

### Downstream consumers

Per the pipeline diagram, `bicycle_route_m_ways_distinct_classified` is the branch point for two further pipelines:
- `05_bicycle_route_infrastructure_per_side`: unnests the table into one row per way per side and assigns the UNECE class / evidence basis / classifiability columns (thesis §3.4.2, §3.4.3).
- Spatial join in stage 06 produces `bicycle_route_m_ways_distinct_classified_per_spatial_unit`, which then feeds the stage-07 extent metrics (`bicycle_route_extent_metrics`) and infrastructure metrics (`bicycle_route_infrastructure_per_side_metrics`).

Neither downstream pipeline modifies this table.

### Table of contents

1. [Environment setup](#1-environment-setup)
2. [Pipeline overview](#2-pipeline-overview)
3. [Step 1: Extraction](#3-step-1-extraction)
4. [Step 2: Normalisation](#4-step-2-normalisation)
5. [Step 3: Mapping style](#5-step-3-mapping-style)
6. [Step 4: Infrastructure classification per side](#6-step-4-infrastructure-classification-per-side)
7. [Step 5: Way-level infrastructure combination](#7-step-5-way-level-infrastructure-combination)
8. [Step 6: Explicitness score](#8-step-6-explicitness-score)
9. [Step 7: Routing surface](#9-step-7-routing-surface)
10. [Step 8: Road directionality](#10-step-8-road-directionality)
11. [Step 9: Bicycle directionality and infrastructure confidence](#11-step-9-bicycle-directionality-and-infrastructure-confidence)
12. [Result: `bicycle_route_m_ways_distinct_classified`](#12-result-bicycle_route_m_ways_distinct_classified)

---

## 1. Environment setup

### Libraries and extensions

In [1]:
from IPython.utils import io
from lonboard import PolygonLayer, Map
from lonboard.colormap import apply_continuous_cmap
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt

### Loading shared variables

Execute the prior notebooks to bring `bicycle_route_m_ways_distinct` and the joined spatial-unit tables into the current session.

In [2]:
with io.capture_output() as captured:
    %run /home/vbo226/03_boundaries_population.ipynb
    %run /home/vbo226/00_bicycle_route_relations.ipynb

`bicycle_route_m_ways_distinct` is now available in the DuckDB session from `00_bicycle_route_relations`. All steps below enrich it with additional columns; no step changes the grain.

In [10]:
# Anchor: all enrichment steps operate on this table
ways = bicycle_route_m_ways_distinct

---

## 2. Pipeline overview

Each step adds columns to the way dataset; the output of every step is still one row per way. The intermediate tables form this chain:

| Step | Output table | Purpose |
|---|---|---|
| 1 | `ways_with_tags` | Raw tag columns extracted from `way_m_tags`. |
| 2 | `ways_normalised` | `effective_cycleway_right` / `effective_cycleway_left` resolved from conflicting tags. |
| 3 | `ways_with_mapping_style` | `mapping_style` assigned. |
| 4 | `ways_with_side_infra` | `way_infrastructure_right` and `way_infrastructure_left`. |
| 5 | `ways_with_infra` | Combined way-level `way_infrastructure`. |
| 6 | `ways_with_access` | `forward_bicycle_access_signal` and `backward_bicycle_access_signal` structs. |
| 7 | `ways_with_surface` | `forward_routing_surface` and `backward_routing_surface`. |
| 8 | `ways_with_road_dir` | `road_directionality` struct. |
| 9 | `bicycle_route_m_ways_distinct_classified` | `bicycle_directionality`, `way_forward_bicycle_access`, `way_backward_bicycle_access`, `infrastructure_confidence`. |

The separation of steps is deliberate. Normalisation is performed once and its output used by all downstream steps, rather than reproducing tag-conflict resolution logic at each step. Routing surface is resolved before directionality because different surfaces carry directionality information in different OSM tag families; resolving the surface first means the directionality step consults exactly the right tags.

---

## 3. Step 1: Extraction

**Purpose.** Pull raw tag values from `way_m_tags` into named columns (`has_bicycle`, `has_bicycle_forward`, `has_bicycle_backward`, `has_vehicle*`, `has_access`, `has_highway`, `has_cycleway*`, `has_oneway*`, `has_cycleway*_oneway`, `has_ferry`). The result, `ways_with_tags`, is the substrate every later step reads from.

In [11]:
ways_with_tags = duckdb.sql(f"""
SELECT *, 

    way_m_tags['bicycle'] as has_bicycle,
    way_m_tags['bicycle:forward'] as has_bicycle_forward,
    way_m_tags['bicycle:backward'] as has_bicycle_backward,
    way_m_tags['vehicle'] as has_vehicle,
    way_m_tags['vehicle:forward'] as has_vehicle_forward,
    way_m_tags['vehicle:backward'] as has_vehicle_backward,
    way_m_tags['access'] as has_access,

    way_m_tags['highway'] as has_highway,
    way_m_tags['cycleway'] as has_cycleway,
    way_m_tags['cycleway:left'] as has_cycleway_left,
    way_m_tags['cycleway:right'] as has_cycleway_right,
    way_m_tags['cycleway:both'] as has_cycleway_both,

    way_m_tags['oneway'] as has_oneway,
    way_m_tags['oneway:bicycle'] as has_oneway_bicycle,
    way_m_tags['cycleway:oneway']     AS has_cycleway_oneway,
    way_m_tags['cycleway:both:oneway']       AS has_cycleway_both_oneway,
    way_m_tags['cycleway:left:oneway']       AS has_cycleway_left_oneway,
    way_m_tags['cycleway:right:oneway']      AS has_cycleway_right_oneway,

    map_contains_entry(way_m_tags, 'route', 'ferry') as has_ferry
    
FROM bicycle_route_m_ways_distinct
""")

---

## 4. Step 2: Normalisation

**Purpose.** Resolve conflicting and redundant `cycleway` tags into a single effective value per side. OSM contributors use these tags inconsistently: some use `cycleway=*` for both sides, others use `cycleway:left` / `cycleway:right` separately, and some combine multiple keys simultaneously with values that may agree or disagree. Without resolving these conflicts here, every downstream step would need to replicate the same priority logic independently.

For each side, a single effective value is selected using the deterministic precedence the thesis records in §3.4.1:

> `cycleway:right` / `cycleway:left` > `cycleway:both` > `cycleway`

Side-specific tags take precedence because they encode more precise spatial information; general tags serve as symmetric fallbacks. The same priority rule is applied to the `cycleway:*:oneway` variants.

**Key property.** All downstream steps depend on `effective_cycleway_right` and `effective_cycleway_left` rather than on the raw tag set, so tag-conflict resolution logic appears exactly once in the pipeline.

In [12]:
ways_normalised = duckdb.sql(f"""
SELECT *,
    -- right side: cycleway:right wins over cycleway:both wins over cycleway
    COALESCE(has_cycleway_right, has_cycleway_both, has_cycleway) AS effective_cycleway_right,
    
    -- left side: cycleway:left wins over cycleway:both wins over cycleway
    COALESCE(has_cycleway_left, has_cycleway_both, has_cycleway)  AS effective_cycleway_left,

    -- oneway right side: cycleway:right:oneway wins over cycleway:both:oneway wins over cycleway:oneway
    COALESCE(has_cycleway_right_oneway, has_cycleway_both_oneway, has_cycleway_oneway) AS effective_cycleway_right_oneway,

    -- oneway left side: cycleway:left:oneway wins over cycleway:both:oneway wins over cycleway:oneway
    COALESCE(has_cycleway_left_oneway, has_cycleway_both_oneway, has_cycleway_oneway)  AS effective_cycleway_left_oneway
FROM ways_with_tags
""")

---

## 5. Step 3: Mapping style

**Purpose.** Classify each way by how its cycling infrastructure is encoded in OSM. The tags used to describe infrastructure, access, and directionality differ by mapping style, so this classification must be established before any of those steps can proceed.

| `mapping_style` | Definition | Operationalisation |
|---|---|---|
| `dedicated_mapping` | Cycling infrastructure mapped as a separate geometry | `highway=cycleway`; `highway=path` + `bicycle=designated` |
| `carriageway_mapping` | Cycling infrastructure encoded as a property of the carriageway | Any `effective_cycleway_right` or `effective_cycleway_left` in the carriageway value set; or `cycleway:lanes=*` variants |
| `n/a` | No carriageway or dedicated cycling-infrastructure tagging present | Ferry routes; ways with no infrastructure tags |
| `tagging_conflict` | Way simultaneously tagged as dedicated infrastructure and carrying carriageway cycleway values | Data-quality issue; carried through pipeline, excluded from confidence classification |

In [13]:
carriageway_mapping_values = ['lane', 'shared_lane', 'share_busway', 'track', 'separate', 'no']

ways_with_mapping_style = duckdb.sql(f"""
SELECT *,
    --Mapping styles
    CASE
        WHEN (
            has_highway = 'cycleway'
            OR (has_highway = 'path' AND has_bicycle = 'designated')
        )
        AND (
            effective_cycleway_left IN {carriageway_mapping_values} 
            OR effective_cycleway_right IN {carriageway_mapping_values} 
        ) THEN 'tagging_conflict'
        WHEN has_ferry IS TRUE                                                                  THEN 'ferry'
        WHEN has_highway = 'cycleway' OR (has_highway = 'path' AND has_bicycle = 'designated') THEN 'dedicated_mapping'
        WHEN effective_cycleway_right IN {carriageway_mapping_values} 
          OR effective_cycleway_left  IN {carriageway_mapping_values}
          OR way_m_tags['cycleway:lanes'] IS NOT NULL 
          OR way_m_tags['cycleway:lanes:forward'] IS NOT NULL 
          OR way_m_tags['cycleway:lanes:backward'] IS NOT NULL                                  THEN 'carriageway_mapping'
        ELSE 'n/a'
    END as mapping_style
FROM ways_normalised
""")

In [14]:
duckdb.sql("""
SELECT 
    mapping_style, 
    COUNT(*) as way_count, 
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_total_ways,
    ROUND(SUM(way_m_length_nl_meters) / 1000, 2) AS total_length_km,
    ROUND(100.0 * SUM(way_m_length_nl_meters)/ SUM(SUM(way_m_length_nl_meters)) OVER (), 2) AS pct_of_total_length
FROM ways_with_mapping_style
GROUP BY mapping_style
ORDER BY total_length_km DESC
""").show(max_width=150)

┌─────────────────────┬───────────┬───────────────────┬─────────────────┬─────────────────────┐
│    mapping_style    │ way_count │ pct_of_total_ways │ total_length_km │ pct_of_total_length │
│       varchar       │   int64   │      double       │     double      │       double        │
├─────────────────────┼───────────┼───────────────────┼─────────────────┼─────────────────────┤
│ n/a                 │     88254 │             48.67 │        19827.34 │                51.4 │
│ dedicated_mapping   │     75646 │             41.72 │        15303.81 │               39.68 │
│ carriageway_mapping │     17222 │               9.5 │          3243.8 │                8.41 │
│ ferry               │       186 │               0.1 │          195.13 │                0.51 │
│ tagging_conflict    │        10 │              0.01 │            0.77 │                 0.0 │
└─────────────────────┴───────────┴───────────────────┴─────────────────┴─────────────────────┘



## 6. Step 4: Infrastructure classification per side

**Purpose.** For `carriageway_mapping` ways, classify each side of the carriageway independently. The per-side split is necessary because cycling infrastructure is frequently asymmetric (present on one side of a road but not the other); a way-level label would obscure these configurations. This is the same motivation the thesis develops at length in §3.4.2 for the per-side step in stage 05.

| Value | Meaning | Raw values triggering it |
|---|---|---|
| `cycling_infrastructure` | Infrastructure physically present on this carriageway side | `lane`, `shared_lane`, `share_busway`, `track` |
| `no_cycling_infrastructure` | Absence explicitly confirmed by a tag | `no` |
| `separate_infrastructure` | Infrastructure exists but mapped as a separate OSM way | `separate` |
| `NULL` | No cycleway tag present for this side; absence not confirmed | *(tag absent)* |
| `n/a` | Per-side classification not applicable | `dedicated_mapping`, ferry |

`NULL` and `no_cycling_infrastructure` are intentionally distinct. `NULL` means the tag is absent and says nothing about whether infrastructure exists; `no_cycling_infrastructure` is an explicit statement that it does not. This distinction is the basis for confidence scoring in Step 9, and for the thesis's `inferred_absence_none` vs `inferred_absence_partial` split in §3.4.4.

In [15]:
carriageway_infrastructure_presence = ['lane', 'shared_lane', 'share_busway', 'track']
carriageway_infrastructure_absent = ['no']
carriageway_infrastructure_separate = ['separate']


ways_with_side_infra = duckdb.sql(f"""
SELECT *,
    --Infrastructure vs no infrastructure
    -- Right side 
    CASE 
        WHEN mapping_style = 'tagging_conflict'                                         THEN 'tagging_conflict'
        WHEN has_ferry IS TRUE THEN 'n/a'
        WHEN has_highway = 'cycleway' OR (has_highway = 'path' AND has_bicycle = 'designated') THEN 'n/a' 
        WHEN effective_cycleway_right IN {carriageway_infrastructure_presence} THEN 'cycling_infrastructure'
        WHEN effective_cycleway_right IN {carriageway_infrastructure_absent}            THEN 'no_cycling_infrastructure'
        WHEN effective_cycleway_right IN {carriageway_infrastructure_separate}           THEN 'separate_infrastructure'
    END as way_infrastructure_right,

    --Left side
    CASE 
        WHEN has_ferry IS TRUE THEN 'n/a'
        WHEN has_highway = 'cycleway' OR (has_highway = 'path' AND has_bicycle = 'designated') THEN 'n/a' 
        WHEN effective_cycleway_left IN {carriageway_infrastructure_presence} THEN 'cycling_infrastructure'
        WHEN effective_cycleway_left IN {carriageway_infrastructure_absent}            THEN 'no_cycling_infrastructure'
        WHEN effective_cycleway_left IN {carriageway_infrastructure_separate}           THEN 'separate_infrastructure'
    END as way_infrastructure_left
FROM ways_with_mapping_style
""")

---

## 7. Step 5: Way-level infrastructure combination

**Purpose.** Combine `way_infrastructure_right` and `way_infrastructure_left` into a single way-level label (`way_infrastructure`) used by the routing surface and confidence steps. The full value set covers three categories of outcome:

- **Symmetric cases**, both sides agree: `symmetric_cycling_infrastructure`, `symmetric_no_cycling_infrastructure`, `symmetric_separate_infrastructure`.
- **True mismatch cases**, both sides tagged with different values: `left_infra_right_no_infra`, `right_infra_left_no_infra`, `left_infra_right_separate`, and so on.
- **One-side-null cases**, one side tagged and the other `NULL`: `right_infra_left_null`, `left_infra_right_null`, `right_no_infra_left_null`, and so on. These drive medium confidence in Step 9.
- **`unexpected_both_null`**: both sides `NULL` for a `carriageway_mapping` way; flagged as a data issue.

In [16]:
ways_with_infra = duckdb.sql("""
SELECT *, 
    CASE
        WHEN mapping_style = 'tagging_conflict'                                         THEN 'tagging_conflict'
        WHEN has_ferry IS TRUE                                                           THEN 'ferry'
        WHEN mapping_style = 'dedicated_mapping'                                        THEN 'cycling_infrastructure'

        -- symmetric cases
        WHEN mapping_style = 'carriageway_mapping'
         AND way_infrastructure_left = 'cycling_infrastructure'
         AND way_infrastructure_right = 'cycling_infrastructure'                         THEN 'symmetric_cycling_infrastructure'

        WHEN mapping_style = 'carriageway_mapping'
         AND way_infrastructure_left = 'no_cycling_infrastructure'
         AND way_infrastructure_right = 'no_cycling_infrastructure'                      THEN 'symmetric_no_cycling_infrastructure'

        WHEN mapping_style = 'carriageway_mapping'
         AND way_infrastructure_left = 'separate_infrastructure'
         AND way_infrastructure_right = 'separate_infrastructure'                        THEN 'symmetric_separate_infrastructure'

        -- true mismatch: both sides tagged but different
        WHEN mapping_style = 'carriageway_mapping'
         AND way_infrastructure_left = 'cycling_infrastructure'
         AND way_infrastructure_right = 'no_cycling_infrastructure'                      THEN 'left_infra_right_no_infra'

        WHEN mapping_style = 'carriageway_mapping'
         AND way_infrastructure_right = 'cycling_infrastructure'
         AND way_infrastructure_left = 'no_cycling_infrastructure'                       THEN 'right_infra_left_no_infra'

        WHEN mapping_style = 'carriageway_mapping'
         AND way_infrastructure_left = 'cycling_infrastructure'
         AND way_infrastructure_right = 'separate_infrastructure'                        THEN 'left_infra_right_separate'

        WHEN mapping_style = 'carriageway_mapping'
         AND way_infrastructure_right = 'cycling_infrastructure'
         AND way_infrastructure_left = 'separate_infrastructure'                         THEN 'right_infra_left_separate'

        WHEN mapping_style = 'carriageway_mapping'
         AND way_infrastructure_left = 'no_cycling_infrastructure'
         AND way_infrastructure_right = 'separate_infrastructure'                        THEN 'left_no_infra_right_separate'

        WHEN mapping_style = 'carriageway_mapping'
         AND way_infrastructure_right = 'no_cycling_infrastructure'
         AND way_infrastructure_left = 'separate_infrastructure'                         THEN 'right_no_infra_left_separate'

        -- one side tagged, other NULL (untagged)
        WHEN mapping_style = 'carriageway_mapping'
         AND way_infrastructure_left IS NULL
         AND way_infrastructure_right = 'cycling_infrastructure'                         THEN 'right_infra_left_null'

        WHEN mapping_style = 'carriageway_mapping'
         AND way_infrastructure_right IS NULL
         AND way_infrastructure_left = 'cycling_infrastructure'                          THEN 'left_infra_right_null'

        WHEN mapping_style = 'carriageway_mapping'
         AND way_infrastructure_left IS NULL
         AND way_infrastructure_right = 'no_cycling_infrastructure'                      THEN 'right_no_infra_left_null'

        WHEN mapping_style = 'carriageway_mapping'
         AND way_infrastructure_right IS NULL
         AND way_infrastructure_left = 'no_cycling_infrastructure'                       THEN 'left_no_infra_right_null'

        WHEN mapping_style = 'carriageway_mapping'
         AND way_infrastructure_left IS NULL
         AND way_infrastructure_right = 'separate_infrastructure'                        THEN 'right_separate_left_null'

        WHEN mapping_style = 'carriageway_mapping'
         AND way_infrastructure_right IS NULL
         AND way_infrastructure_left = 'separate_infrastructure'                         THEN 'left_separate_right_null'

        -- both null: should not occur for carriageway_mapping, flag as data issue
        WHEN mapping_style = 'carriageway_mapping'
         AND way_infrastructure_left IS NULL
         AND way_infrastructure_right IS NULL                                             THEN 'unexpected_both_null'

        ELSE NULL
    END AS way_infrastructure
FROM ways_with_side_infra 
""")

In [17]:
duckdb.sql("""
SELECT 
    mapping_style,
    way_infrastructure,
    COUNT(*) AS n_ways,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_total_ways,
    ROUND(SUM(way_m_length_nl_meters) / 1000, 2) AS total_length_km,
    ROUND(100.0 * SUM(way_m_length_nl_meters)/ SUM(SUM(way_m_length_nl_meters)) OVER (), 2) AS pct_of_total_length
FROM ways_with_infra
GROUP BY 1, 2
ORDER BY total_length_km DESC
""").show()

┌─────────────────────┬───────────────────────────┬────────┬───────────────────┬─────────────────┬─────────────────────┐
│    mapping_style    │    way_infrastructure     │ n_ways │ pct_of_total_ways │ total_length_km │ pct_of_total_length │
│       varchar       │          varchar          │ int64  │      double       │     double      │       double        │
├─────────────────────┼───────────────────────────┼────────┼───────────────────┼─────────────────┼─────────────────────┤
│ n/a                 │ NULL                      │  88254 │             48.67 │        19827.34 │                51.4 │
│ dedicated_mapping   │ cycling_infrastructure    │  75646 │             41.72 │        15303.81 │               39.68 │
│ carriageway_mapping │ symmetric_cycling_infras… │   9310 │              5.13 │         1894.84 │                4.91 │
│ carriageway_mapping │ symmetric_no_cycling_inf… │   5210 │              2.87 │         1186.28 │                3.08 │
│ ferry               │ ferry   

## 8. Step 6: Explicitness score

**Purpose.** Determine, for each direction of travel, what bicycle access value applies and how directly it is supported by OSM tags. The score reflects strength of evidence: not whether cycling is permitted, but how confidently that can be concluded from available tags. The thesis (§3.4.3, §3.4.4) uses this score downstream when distinguishing recorded presence, recorded explicit absence, and inferred absence.

The resolution follows a tag priority hierarchy per direction, falling back from the most explicit to the most inferred:

| Source | Score | Example |
|---|---|---|
| Explicit bicycle tag (`bicycle=*`, directional variants) | 1.00 | `bicycle=yes` |
| Vehicle-level tag (`vehicle=*`, directional variants) | 0.95 | `vehicle=no` |
| General access tag (`access=*`, directional variants) | 0.90 | `access=yes` |
| Carriageway cycling infrastructure present (inferred) | 0.75 | `effective_cycleway_right=lane` |
| Highway-type legal default (Netherlands OSM assumptions) | 0.20–0.85 | `highway=residential` → `yes` |

Each direction produces a struct `{value, source, explicitness}`..

In [18]:
ways_with_access = duckdb.sql("""
WITH legal_defaults AS (
SELECT *, 
    CASE
        WHEN has_highway = 'cycleway' THEN {value: 'designated', score: 0.85}
                
        WHEN has_highway = 'tertiary' THEN {value:'yes', score: 0.65} --implies access=yes
        WHEN has_highway = 'tertiary_link' THEN {value:'yes', score: 0.65}      
        WHEN has_highway = 'residential' THEN {value:'yes', score: 0.65} --implies access=yes
        WHEN has_highway = 'unclassified' THEN {value:'yes', score: 0.60} 
        WHEN has_highway = 'secondary' THEN {value:'yes', score: 0.55} --implies motorcar=yes
        WHEN has_highway = 'secondary_link' THEN {value:'yes', score: 0.55}     
        WHEN has_highway = 'service' THEN {value:'yes', score: 0.60}
        WHEN has_highway = 'living_street' THEN {value:'yes', score: 0.60}
        
        WHEN has_highway = 'busway' THEN {value:'no', score: 0.65} --implies access=no
        WHEN has_highway = 'bridleway' THEN {value:'no', score: 0.20} --implies bicycle=yes
        
        WHEN has_highway = 'pedestrian' THEN {value:'dismount', score: 0.60}
        WHEN has_highway = 'footway' THEN {value:'dismount', score: 0.20}

        WHEN has_highway = 'primary' THEN {value:'yes', score: 0.20}
        WHEN has_highway = 'path' THEN {value:'yes', score: 0.20}
        WHEN has_highway = 'track' THEN {value:'yes', score: 0.20}

        WHEN has_highway = 'steps' THEN {value:'dismount', score: 0.20} --implies access=no
        WHEN has_highway = 'construction' THEN {value:'N/A', score: -999.9}
        WHEN has_highway = 'proposed' THEN {value:'N/A', score: -999.9}
        WHEN has_highway = 'no' THEN {value:'N/A', score: -999.9}
        
    END as highway_legal_defaults 
FROM ways_with_infra
)
SELECT *,
    -- forward resolution
    CASE
        WHEN mapping_style = 'tagging_conflict' THEN {value: 'N/A', source: 'tagging_conflict', explicitness: NULL}
        WHEN has_bicycle_forward IS NOT NULL
            THEN {value: has_bicycle_forward, source: 'explicit_bicycle_forward', explicitness: 1.0}
        WHEN has_bicycle IS NOT NULL
            THEN {value: has_bicycle, source: 'explicit_bicycle', explicitness: 1.0}
        WHEN has_vehicle_forward IS NOT NULL
            THEN {value: has_vehicle_forward, source: 'explicit_vehicle_forward', explicitness: 0.95}
        WHEN has_vehicle IS NOT NULL
            THEN {value: has_vehicle, source: 'explicit_vehicle', explicitness: 0.95}
        WHEN has_access IS NOT NULL
            THEN {value: has_access, source: 'explicit_access', explicitness: 0.90}

    -- NOTE: this step depends on normalization having already resolved tag conflicts
    -- (cycleway:right > cycleway:both > cycleway) into effective_cycleway_right/left,
    -- and having classified mapping_style and infrastructure_category at way level.
    -- Without normalization, the infrastructure tier below would need to replicate
    -- conflict resolution logic inline for every cycleway tag combination —
    -- instead we can make a clean decision on a single pre-resolved category.

        WHEN way_infrastructure_right = 'cycling_infrastructure' 
         AND mapping_style = 'carriageway_mapping'
            THEN {value: 'yes', source: 'carriageway_infrastructure', explicitness: 0.75}
        ELSE {value: highway_legal_defaults.value, source: 'road_type_default', explicitness: highway_legal_defaults.score}
    END AS forward_bicycle_access_signal,

    -- backward resolution
    CASE
        WHEN mapping_style = 'tagging_conflict' THEN {value: 'N/A', source: 'tagging_conflict', explicitness: NULL}
        WHEN has_bicycle_backward IS NOT NULL
            THEN {value: has_bicycle_backward, source: 'explicit_bicycle_backward', explicitness: 1.0}
        WHEN has_bicycle IS NOT NULL
            THEN {value: has_bicycle, source: 'explicit_bicycle', explicitness: 1.0}
        WHEN has_vehicle_backward IS NOT NULL
            THEN {value: has_vehicle_backward, source: 'explicit_vehicle_backward', explicitness: 0.95}
        WHEN has_vehicle IS NOT NULL
            THEN {value: has_vehicle, source: 'explicit_vehicle', explicitness: 0.95}
        WHEN has_access IS NOT NULL
            THEN {value: has_access, source: 'explicit_access', explicitness: 0.90}
        WHEN way_infrastructure_left = 'cycling_infrastructure' 
        AND mapping_style = 'carriageway_mapping'
            THEN {value: 'yes', source: 'carriageway_infrastructure', explicitness: 0.75}
        ELSE {value: highway_legal_defaults.value, source: 'road_type_default', explicitness: highway_legal_defaults.score}
    END AS backward_bicycle_access_signal
FROM legal_defaults
""")

In [19]:
# Distribution of access signal sources — forward direction
duckdb.sql("""
SELECT
    forward_bicycle_access_signal.source  AS forward_source,
    COUNT(*)                              AS n_ways,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_ways
FROM ways_with_access
GROUP BY 1
ORDER BY n_ways DESC
""")

┌────────────────────────────┬────────┬──────────┐
│       forward_source       │ n_ways │ pct_ways │
│          varchar           │ int64  │  double  │
├────────────────────────────┼────────┼──────────┤
│ road_type_default          │ 161615 │    89.13 │
│ carriageway_infrastructure │  11096 │     6.12 │
│ explicit_bicycle           │   8074 │     4.45 │
│ explicit_access            │    290 │     0.16 │
│ explicit_bicycle_forward   │    164 │     0.09 │
│ explicit_vehicle           │     68 │     0.04 │
│ tagging_conflict           │     10 │     0.01 │
│ explicit_vehicle_forward   │      1 │      0.0 │
└────────────────────────────┴────────┴──────────┘

---

## 9. Step 7: Routing surface

**Purpose.** Determine which physical surface a cyclist uses on each way, for each direction of travel. Resolving this before directionality means the directionality step knows exactly which tag family to consult, rather than checking all families simultaneously and handling conflicts between them.

| `routing_surface` value | Condition | Cyclist uses |
|---|---|---|
| `carriageway_infrastructure` | `carriageway_mapping` + `cycling_infrastructure` on relevant side | Lane or track alongside carriageway |
| `carriageway` | No cycleway tags, explicit `no_cycling_infrastructure`, or `NULL` on relevant side | Road surface |
| `separate_surface` | `separate_infrastructure` on relevant side, or `use_sidepath` in access signal | Separately mapped OSM way; this way contributes `access=no` for this direction |
| `dedicated_surface` | `dedicated_mapping` | The way itself |
| `ferry` | `has_ferry IS TRUE` | Vessel |

`use_sidepath` is checked before `separate_infrastructure` because it is an explicit access statement (score 1.00) and overrides infrastructure inference.

In [20]:
ways_with_surface = duckdb.sql("""
SELECT *,
    CASE
        -- tagging conflict: cannot determine routing surface
        WHEN mapping_style = 'tagging_conflict'
            THEN 'tagging_conflict'

        -- ferry
        WHEN has_ferry IS TRUE
            THEN 'ferry'

        -- dedicated mapping: way itself is the cycling surface, sides not applicable
        WHEN mapping_style = 'dedicated_mapping'
            THEN 'dedicated_surface'

        -- explicit use_sidepath overrides infrastructure presence
        -- wiki states use_sidepath should not be combined with cycleway=lane/track
        -- unless a separate geometry exists — respect explicit signal over inferred infrastructure
        WHEN forward_bicycle_access_signal.value = 'use_sidepath'
            THEN 'separate_surface'

        -- forward approximated as right side
        -- carriageway infrastructure present on right side
        WHEN way_infrastructure_right = 'cycling_infrastructure'
            THEN 'carriageway_infrastructure'

        -- separate infrastructure pointer on right side
        WHEN way_infrastructure_right = 'separate_infrastructure'
            THEN 'separate_surface'

        -- no infrastructure on right side, explicitly confirmed or untagged
        WHEN way_infrastructure_right = 'no_cycling_infrastructure'
            THEN 'carriageway'

        -- right side untagged, no cycling infrastructure context
        WHEN way_infrastructure_right IS NULL
            THEN 'carriageway'

        ELSE 'carriageway'
    END AS forward_routing_surface,

    CASE
        -- tagging conflict: cannot determine routing surface
        WHEN mapping_style = 'tagging_conflict'
            THEN 'tagging_conflict'

        -- ferry
        WHEN has_ferry IS TRUE
            THEN 'ferry'

        -- dedicated mapping
        WHEN mapping_style = 'dedicated_mapping'
            THEN 'dedicated_surface'

        -- explicit use_sidepath overrides infrastructure presence
        WHEN backward_bicycle_access_signal.value = 'use_sidepath'
            THEN 'separate_surface'

        -- backward approximated as left side
        -- carriageway infrastructure present on left side
        WHEN way_infrastructure_left = 'cycling_infrastructure'
            THEN 'carriageway_infrastructure'

        -- separate infrastructure pointer on left side
        WHEN way_infrastructure_left = 'separate_infrastructure'
            THEN 'separate_surface'

        -- no infrastructure on left side, explicitly confirmed or untagged
        WHEN way_infrastructure_left = 'no_cycling_infrastructure'
            THEN 'carriageway'

        -- left side untagged, no cycling infrastructure context
        WHEN way_infrastructure_left IS NULL
            THEN 'carriageway'

        ELSE 'carriageway'
    END AS backward_routing_surface

FROM ways_with_access
""")

In [21]:
# Distribution of routing surface combinations
duckdb.sql("""
SELECT
    forward_routing_surface,
    backward_routing_surface,
    COUNT(*)                                                AS n_ways,
    ROUND(SUM(way_m_length_nl_meters) / 1000, 1)              AS total_km,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2)     AS pct_ways
FROM ways_with_surface
GROUP BY 1, 2
ORDER BY n_ways DESC
""").show(max_width=150)

┌────────────────────────────┬────────────────────────────┬────────┬──────────┬──────────┐
│  forward_routing_surface   │  backward_routing_surface  │ n_ways │ total_km │ pct_ways │
│          varchar           │          varchar           │ int64  │  double  │  double  │
├────────────────────────────┼────────────────────────────┼────────┼──────────┼──────────┤
│ carriageway                │ carriageway                │  93386 │  21011.3 │     51.5 │
│ dedicated_surface          │ dedicated_surface          │  75646 │  15303.8 │    41.72 │
│ carriageway_infrastructure │ carriageway_infrastructure │   9299 │   1894.1 │     5.13 │
│ carriageway_infrastructure │ carriageway                │   2088 │    117.9 │     1.15 │
│ carriageway                │ carriageway_infrastructure │    255 │     24.9 │     0.14 │
│ ferry                      │ ferry                      │    186 │    195.1 │      0.1 │
│ carriageway_infrastructure │ separate_surface           │    162 │     10.4 │     0.09 │

---

## 10. Step 8: Road directionality

**Purpose.** Determine the physical direction of travel permitted on the way from `oneway=*`. This step is narrow by design: it reads only `oneway=*` and produces one of four values. Cycling-specific overrides (`oneway:bicycle=*`) are left entirely to Step 9, keeping road directionality (what the road physically is) separate from bicycle directionality (what that means for cyclists specifically).

The output is `road_directionality`, a struct `{value, source}` whose `value` is one of `oneway_forward`, `oneway_backward`, `bidirectional`, `alternating`, or `N/A`.

In [22]:
ways_with_road_dir = duckdb.sql("""
SELECT *,
    CASE
        WHEN mapping_style = 'tagging_conflict' THEN {value: 'N/A', source: 'tagging_conflict'}
        WHEN has_ferry IS TRUE THEN
            CASE
                WHEN has_oneway = 'yes'         THEN {value: 'oneway_forward',  source: 'explicit_oneway'}
                WHEN has_oneway = '-1'          THEN {value: 'oneway_backward', source: 'explicit_oneway'}
                WHEN has_oneway = 'no'          THEN {value: 'bidirectional',   source: 'explicit_bidirectional'}
                WHEN has_oneway = 'alternating' THEN {value: 'alternating',     source: 'explicit_alternating'}
                ELSE                                 {value: 'bidirectional',   source: 'implicit_bidirectional'}
            END
        WHEN forward_routing_surface = 'dedicated_surface'
          OR backward_routing_surface = 'dedicated_surface' THEN
            CASE
                WHEN has_oneway = 'yes'         THEN {value: 'oneway_forward',  source: 'explicit_oneway'}
                WHEN has_oneway = '-1'          THEN {value: 'oneway_backward', source: 'explicit_oneway'}
                WHEN has_oneway = 'no'          THEN {value: 'bidirectional',   source: 'explicit_bidirectional'}
                WHEN has_oneway = 'alternating' THEN {value: 'alternating',     source: 'explicit_alternating'}
                ELSE                                 {value: 'bidirectional',   source: 'implicit_bidirectional'}
            END
        ELSE
            CASE
                WHEN has_oneway = 'yes'         THEN {value: 'oneway_forward',  source: 'explicit_oneway'}
                WHEN has_oneway = '-1'          THEN {value: 'oneway_backward', source: 'explicit_oneway'}
                WHEN has_oneway = 'no'          THEN {value: 'bidirectional',   source: 'explicit_bidirectional'}
                WHEN has_oneway = 'alternating' THEN {value: 'alternating',     source: 'explicit_alternating'}
                ELSE                                 {value: 'bidirectional',   source: 'implicit_bidirectional'}
            END
    END AS road_directionality
FROM ways_with_surface
""")

In [23]:
# Distribution of road directionality values
duckdb.sql("""
SELECT
    road_directionality.value   AS directionality,
    road_directionality.source  AS source,
    COUNT(*)                    AS n_ways,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_ways
FROM ways_with_road_dir
GROUP BY 1, 2
ORDER BY n_ways DESC
""").show(max_width=150)

┌────────────────┬────────────────────────┬────────┬──────────┐
│ directionality │         source         │ n_ways │ pct_ways │
│    varchar     │        varchar         │ int64  │  double  │
├────────────────┼────────────────────────┼────────┼──────────┤
│ bidirectional  │ implicit_bidirectional │ 130276 │    71.85 │
│ oneway_forward │ explicit_oneway        │  27780 │    15.32 │
│ bidirectional  │ explicit_bidirectional │  23240 │    12.82 │
│ alternating    │ explicit_alternating   │     12 │     0.01 │
│ N/A            │ tagging_conflict       │     10 │     0.01 │
└────────────────┴────────────────────────┴────────┴──────────┘



---

## 11. Step 9: Bicycle directionality and infrastructure confidence

This step produces two outputs in a single pass and closes the pipeline; the grain remains one row per way.

**Bicycle directionality** applies cycling-specific overrides to road directionality. The tags consulted depend on the routing surface:

| Routing surface | Directionality tags consulted |
|---|---|
| `carriageway` | `oneway:bicycle=*` |
| `carriageway_infrastructure` | `cycleway:right:oneway=*` / `cycleway:left:oneway=*` |
| `dedicated_surface` | `oneway=*` / `cycleway:oneway=*` |
| `separate_surface` | Not applicable; the cyclist uses a different OSM way |
| `ferry` | `oneway=*` |

**Infrastructure confidence** classifies each way by how completely cycling infrastructure is described by available tags, drawing on `way_infrastructure` and contextual signals (`use_sidepath`, `junction=roundabout`, `mapping_style`):

| `infrastructure_confidence` | Meaning |
|---|---|
| `certain` | Both sides fully tagged, or dedicated infrastructure. No ambiguity. |
| `high_explained_by_sidepath` | One side tagged; other `NULL`, but `use_sidepath` explains the asymmetry. |
| `high_explained_by_roundabout` | One side tagged; other `NULL`, but `junction=roundabout` explains the asymmetry. |
| `medium_side_null_unexplained` | One side tagged; other `NULL` with no contextual explanation. |
| `high_use_sidepath_confirms_no_infrastructure` | `n/a` mapping style but `use_sidepath` confirms no carriageway infrastructure. |
| `low_no_cycleway_infrastructure_signal` | No cycleway tags and no `use_sidepath`. Infrastructure cannot be determined. |
| `tagging_conflict` | Conflicting tags. Excluded from confidence-based analysis. |
| `ferry` | Ferry route. |

In [32]:
ways_with_bicycle_dir = duckdb.sql("""
WITH bicycle_directionality_cte AS (
SELECT *,
    -- forward bicycle directionality
    CASE
        WHEN has_ferry IS TRUE
            THEN {value: road_directionality.value, source: 'ferry_follows_road'}

        WHEN forward_routing_surface = 'separate_surface'
            THEN {value: 'N/A', source: 'separate_surface_not_resolved_here'}

        WHEN forward_routing_surface = 'tagging_conflict'
            THEN {value: 'N/A', source: 'tagging_conflict'}

        WHEN forward_routing_surface = 'dedicated_surface' THEN
            CASE
                WHEN road_directionality.value = 'bidirectional'
                    THEN {value: 'both',            source: 'inherit_bidirectional_road'}
                WHEN road_directionality.value = 'oneway_forward'
                    THEN {value: 'oneway_forward',  source: 'inherit_oneway_road'}
                WHEN road_directionality.value = 'oneway_backward'
                    THEN {value: 'oneway_backward', source: 'inherit_oneway_road'}
                WHEN road_directionality.value = 'alternating'
                    THEN {value: 'alternating',     source: 'inherit_alternating_road'}
                ELSE     {value: 'N/A',             source: 'unknown'}
            END

        WHEN forward_routing_surface = 'carriageway_infrastructure' THEN
            CASE
                -- explicit cycleway oneway tags
                WHEN effective_cycleway_right_oneway = 'yes'
                    THEN {value: 'oneway_forward',  source: 'explicit_cycleway_right_oneway'}
                WHEN effective_cycleway_right_oneway = '-1'
                    THEN {value: 'oneway_backward', source: 'explicit_cycleway_right_oneway'}
                WHEN effective_cycleway_right_oneway = 'no'
                    THEN {value: 'both',            source: 'explicit_cycleway_right_bidirectional'}
                -- fall back to road directionality
                WHEN road_directionality.value = 'bidirectional'
                    THEN {value: 'both',            source: 'inherit_bidirectional_road'}
                WHEN road_directionality.value = 'oneway_forward'
                    THEN {value: 'oneway_forward',  source: 'inherit_oneway_road'}
                WHEN road_directionality.value = 'oneway_backward'
                    THEN {value: 'oneway_backward', source: 'inherit_oneway_road'}
                WHEN road_directionality.value = 'alternating'
                    THEN {value: 'alternating',     source: 'inherit_alternating_road'}
                ELSE     {value: 'N/A',             source: 'unknown'}
            END

        -- carriageway: apply oneway:bicycle overrides
        WHEN forward_routing_surface = 'carriageway' THEN
            CASE
                WHEN road_directionality.value = 'bidirectional' AND has_oneway_bicycle IS NOT NULL
                    THEN {value: 'N/A',             source: 'tagging_error_oneway_bicycle_on_bidirectional'}
                WHEN road_directionality.value = 'bidirectional'
                    THEN {value: 'both',            source: 'inherit_bidirectional_road'}
                WHEN road_directionality.value = 'oneway_forward'  AND has_oneway_bicycle = 'no'
                    THEN {value: 'both',            source: 'explicit_exemption_oneway'}
                WHEN road_directionality.value = 'oneway_forward'  AND has_oneway_bicycle = 'yes'
                    THEN {value: 'oneway_forward',  source: 'explicit_follow_oneway'}
                WHEN road_directionality.value = 'oneway_forward'
                    THEN {value: 'oneway_forward',  source: 'inherit_oneway_road'}
                WHEN road_directionality.value = 'oneway_backward' AND has_oneway_bicycle = 'no'
                    THEN {value: 'both',            source: 'explicit_exemption_oneway'}
                WHEN road_directionality.value = 'oneway_backward' AND has_oneway_bicycle = 'yes'
                    THEN {value: 'oneway_backward', source: 'explicit_follow_oneway'}
                WHEN road_directionality.value = 'oneway_backward'
                    THEN {value: 'oneway_backward', source: 'inherit_oneway_road'}
                WHEN road_directionality.value = 'alternating' AND has_oneway_bicycle IS NOT NULL
                    THEN {value: 'N/A',             source: 'tagging_error_oneway_bicycle_on_alternating'}
                WHEN road_directionality.value = 'alternating'
                    THEN {value: 'alternating',     source: 'inherit_alternating_road'}
                ELSE     {value: 'N/A',             source: 'unknown'}
            END

        ELSE {value: 'N/A', source: 'unhandled_routing_surface'}
    END AS forward_bicycle_directionality,

    -- backward bicycle directionality
    CASE
        WHEN has_ferry IS TRUE
            THEN {value: road_directionality.value, source: 'ferry_follows_road'}

        WHEN backward_routing_surface = 'separate_surface'
            THEN {value: 'N/A', source: 'separate_surface_not_resolved_here'}

        WHEN backward_routing_surface = 'tagging_conflict'
            THEN {value: 'N/A', source: 'tagging_conflict'}

        WHEN backward_routing_surface = 'dedicated_surface' THEN
            CASE
                WHEN road_directionality.value = 'bidirectional'
                    THEN {value: 'both',            source: 'inherit_bidirectional_road'}
                WHEN road_directionality.value = 'oneway_forward'
                    THEN {value: 'oneway_forward',  source: 'inherit_oneway_road'}
                WHEN road_directionality.value = 'oneway_backward'
                    THEN {value: 'oneway_backward', source: 'inherit_oneway_road'}
                WHEN road_directionality.value = 'alternating'
                    THEN {value: 'alternating',     source: 'inherit_alternating_road'}
                ELSE     {value: 'N/A',             source: 'unknown'}
            END

        WHEN backward_routing_surface = 'carriageway_infrastructure' THEN
            CASE
                WHEN effective_cycleway_left_oneway = 'yes'
                    THEN {value: 'oneway_forward',  source: 'explicit_cycleway_left_oneway'}
                WHEN effective_cycleway_left_oneway = '-1'
                    THEN {value: 'oneway_backward', source: 'explicit_cycleway_left_oneway'}
                WHEN effective_cycleway_left_oneway = 'no'
                    THEN {value: 'both',            source: 'explicit_cycleway_left_bidirectional'}
                WHEN road_directionality.value = 'bidirectional'
                    THEN {value: 'both',            source: 'inherit_bidirectional_road'}
                WHEN road_directionality.value = 'oneway_forward'
                    THEN {value: 'oneway_forward',  source: 'inherit_oneway_road'}
                WHEN road_directionality.value = 'oneway_backward'
                    THEN {value: 'oneway_backward', source: 'inherit_oneway_road'}
                WHEN road_directionality.value = 'alternating'
                    THEN {value: 'alternating',     source: 'inherit_alternating_road'}
                ELSE     {value: 'N/A',             source: 'unknown'}
            END

        WHEN backward_routing_surface = 'carriageway' THEN
            CASE
                WHEN road_directionality.value = 'bidirectional' AND has_oneway_bicycle IS NOT NULL
                    THEN {value: 'N/A',             source: 'tagging_error_oneway_bicycle_on_bidirectional'}
                WHEN road_directionality.value = 'bidirectional'
                    THEN {value: 'both',            source: 'inherit_bidirectional_road'}
                WHEN road_directionality.value = 'oneway_forward'  AND has_oneway_bicycle = 'no'
                    THEN {value: 'both',            source: 'explicit_exemption_oneway'}
                WHEN road_directionality.value = 'oneway_forward'  AND has_oneway_bicycle = 'yes'
                    THEN {value: 'oneway_forward',  source: 'explicit_follow_oneway'}
                WHEN road_directionality.value = 'oneway_forward'
                    THEN {value: 'oneway_forward',  source: 'inherit_oneway_road'}
                WHEN road_directionality.value = 'oneway_backward' AND has_oneway_bicycle = 'no'
                    THEN {value: 'both',            source: 'explicit_exemption_oneway'}
                WHEN road_directionality.value = 'oneway_backward' AND has_oneway_bicycle = 'yes'
                    THEN {value: 'oneway_backward', source: 'explicit_follow_oneway'}
                WHEN road_directionality.value = 'oneway_backward'
                    THEN {value: 'oneway_backward', source: 'inherit_oneway_road'}
                WHEN road_directionality.value = 'alternating' AND has_oneway_bicycle IS NOT NULL
                    THEN {value: 'N/A',             source: 'tagging_error_oneway_bicycle_on_alternating'}
                WHEN road_directionality.value = 'alternating'
                    THEN {value: 'alternating',     source: 'inherit_alternating_road'}
                ELSE     {value: 'N/A',             source: 'unknown'}
            END

        ELSE {value: 'N/A', source: 'unhandled_routing_surface'}
    END AS backward_bicycle_directionality
FROM ways_with_road_dir
)
SELECT *,
    -- forward access
    CASE
        WHEN forward_routing_surface = 'separate_surface'
            THEN 'no'           -- carriageway closed, cyclist uses separate geometry
        WHEN forward_routing_surface = 'tagging_conflict'
            THEN 'N/A'
        WHEN forward_routing_surface = 'ferry' AND road_directionality.value IN ('bidirectional', 'oneway_forward', 'alternating')
            THEN forward_bicycle_access_signal.value
        WHEN forward_routing_surface = 'ferry'
            THEN 'no'
        WHEN forward_bicycle_directionality.value = 'N/A'
            THEN 'N/A'
        WHEN forward_bicycle_directionality.value IN ('both', 'oneway_forward', 'alternating')
            THEN forward_bicycle_access_signal.value
        ELSE 'no'
    END AS way_forward_bicycle_access,

    -- backward access
    CASE
        WHEN backward_routing_surface = 'separate_surface'
            THEN 'no'           -- carriageway closed, cyclist uses separate geometry
        WHEN backward_routing_surface = 'tagging_conflict'
            THEN 'N/A'
        WHEN backward_routing_surface = 'ferry' AND road_directionality.value IN ('bidirectional', 'oneway_backward', 'alternating')
            THEN backward_bicycle_access_signal.value
        WHEN backward_routing_surface = 'ferry'
            THEN 'no'
        WHEN backward_bicycle_directionality.value = 'N/A'
            THEN 'N/A'
        WHEN backward_bicycle_directionality.value IN ('both', 'oneway_backward', 'alternating')
            THEN backward_bicycle_access_signal.value
        ELSE 'no'
    END AS way_backward_bicycle_access

FROM bicycle_directionality_cte
""")


In [25]:
bicycle_route_m_ways_distinct_classified  = duckdb.sql("""
SELECT *, 
    CASE 
            -- ferry
        WHEN mapping_style = 'ferry'
            THEN 'ferry'
    
        -- tagging conflict: tagging_conflict
        WHEN mapping_style = 'tagging_conflict'
            THEN 'tagging_conflict'
    
        -- dedicated mapping: certain by definition
        -- highway=cycleway or highway=path + bicycle=designated
        -- the way itself IS the infrastructure
        WHEN mapping_style = 'dedicated_mapping'
            THEN 'certain'
            
    --> Certain of presence/absence of cycling infrastructure 
        WHEN way_infrastructure IN (
        'symmetric_cycling_infrastructure',  -- both sides explicitly tagged with infrastructure present
        'symmetric_no_cycling_infrastructure', -- both sides explicitly tagged with infrastructure absent
        'symmetric_separate_infrastructure', -- separate infrastructure explicitly confirmed on both sides

        'right_infra_left_no_infra',    -- one side infra, other explicitly absent — both sides stated
        'left_infra_right_no_infra',

        'right_infra_left_separate',  -- one side infra, other explicitly separate — both sides stated
        'left_infra_right_separate',
  
        'right_no_infra_left_separate',  -- no infra on one side, separate on other — both sides stated
        'left_no_infra_right_separate' 
        )
        THEN 'certain'

    --> High confidence of presence/absence of cycling infrastructure
        WHEN way_infrastructure IN ('right_infra_left_null', -- one side has infrastructure, other is NULL but use_sidepath present
                                    'left_infra_right_null', 
                                    
                                    'right_no_infra_left_null', -- no infra on one side, other NULL but use_sidepath present
                                    'left_no_infra_right_null',

                                    'right_separate_left_null', -- separate on one side, other NULL but use_sidepath present
                                    'left_separate_right_null',

                                    'right_separate_left_null',  -- separate on one side, other NULL but use_sidepath present
                                    'left_separate_right_null',
                                    )
         AND (way_m_tags['bicycle:forward'] = 'use_sidepath' OR way_m_tags['bicycle:backward'] = 'use_sidepath' OR way_m_tags['bicycle'] = 'use_sidepath')
        THEN 'high_explained_by_sidepath'

        WHEN way_infrastructure IN ('right_infra_left_null', -- one side has infrastructure, other is NULL but use_sidepath present
                                    'left_infra_right_null', 
                                    
                                    'right_no_infra_left_null', -- no infra on one side, other NULL but use_sidepath present
                                    'left_no_infra_right_null',

                                    'right_separate_left_null', -- separate on one side, other NULL but use_sidepath present
                                    'left_separate_right_null',

                                    'right_separate_left_null',  -- separate on one side, other NULL but use_sidepath present
                                    'left_separate_right_null',
                                    )
        AND way_m_tags['junction'] = 'roundabout'
        THEN 'high_explained_by_roundabout'
                                    
    --> Medium confidence of presence/absence of cycling infrastructure
        WHEN way_infrastructure IN ('right_infra_left_null', -- one side has infrastructure, other is NULL but use_sidepath present
                    'left_infra_right_null', 
                    
                    'right_no_infra_left_null', -- no infra on one side, other NULL but use_sidepath present
                    'left_no_infra_right_null',

                    'right_separate_left_null', -- separate on one side, other NULL but use_sidepath present
                    'left_separate_right_null',

                    'right_separate_left_null',  -- separate on one side, other NULL but use_sidepath present
                    'left_separate_right_null'
                    )
         AND NOT (
                COALESCE(way_m_tags['bicycle:forward'], '') = 'use_sidepath'
             OR COALESCE(way_m_tags['bicycle:backward'], '') = 'use_sidepath'
             OR COALESCE(way_m_tags['bicycle'], '') = 'use_sidepath'
            )
        THEN 'medium_side_null_unexplained'

    -- n/a: no cycleway tags — handle separately 
        -- use_sidepath on n/a way: separate geometry confirmed
        -- carriageway infrastructure absence explicitly signalled
    WHEN mapping_style = 'n/a'
     AND (way_m_tags['bicycle:forward']  = 'use_sidepath'
       OR way_m_tags['bicycle:backward'] = 'use_sidepath'
       OR way_m_tags['bicycle']          = 'use_sidepath')
        THEN 'high_use_sidepath_confirms_no_infrastructure'

    -- n/a with no supporting signal: genuinely ambiguous
    WHEN mapping_style = 'n/a'
        THEN 'low_no_cycleway_infrastructure_signal'

    ELSE 'unhandled'
        
    END as infrastructure_confidence
FROM ways_with_bicycle_dir
""")

In [26]:
# Distribution of infrastructure_confidence values
duckdb.sql("""
SELECT
    infrastructure_confidence,
    COUNT(*)                                                AS n_ways,
    ROUND(SUM(way_m_length_nl_meters) / 1000, 1)              AS total_km,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2)     AS pct_ways
FROM bicycle_route_m_ways_distinct_classified 
GROUP BY 1
ORDER BY n_ways DESC
""")

┌──────────────────────────────────────────────┬────────┬──────────┬──────────┐
│          infrastructure_confidence           │ n_ways │ total_km │ pct_ways │
│                   varchar                    │ int64  │  double  │  double  │
├──────────────────────────────────────────────┼────────┼──────────┼──────────┤
│ certain                                      │  90411 │  18404.3 │    49.86 │
│ low_no_cycleway_infrastructure_signal        │  88078 │  19821.1 │    48.58 │
│ medium_side_null_unexplained                 │   1654 │    122.1 │     0.91 │
│ high_explained_by_roundabout                 │    606 │      9.7 │     0.33 │
│ high_explained_by_sidepath                   │    197 │     11.4 │     0.11 │
│ ferry                                        │    186 │    195.1 │      0.1 │
│ high_use_sidepath_confirms_no_infrastructure │    176 │      6.3 │      0.1 │
│ tagging_conflict                             │     10 │      0.8 │     0.01 │
└───────────────────────────────────────

In [27]:
duckdb.sql("""
SELECT
    mapping_style,
    infrastructure_confidence,
    COUNT(*) as way_count, 
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_total_ways,
    ROUND(SUM(way_m_length_nl_meters) / 1000, 2) AS total_length_km,
    ROUND(100.0 * SUM(way_m_length_nl_meters)/ SUM(SUM(way_m_length_nl_meters)) OVER (), 2) AS pct_of_total_length
FROM bicycle_route_m_ways_distinct_classified
GROUP BY mapping_style, infrastructure_confidence
ORDER BY total_length_km DESC
""").show(max_width=180)

┌─────────────────────┬──────────────────────────────────────────────┬───────────┬───────────────────┬─────────────────┬─────────────────────┐
│    mapping_style    │          infrastructure_confidence           │ way_count │ pct_of_total_ways │ total_length_km │ pct_of_total_length │
│       varchar       │                   varchar                    │   int64   │      double       │     double      │       double        │
├─────────────────────┼──────────────────────────────────────────────┼───────────┼───────────────────┼─────────────────┼─────────────────────┤
│ n/a                 │ low_no_cycleway_infrastructure_signal        │     88078 │             48.58 │        19821.06 │               51.39 │
│ dedicated_mapping   │ certain                                      │     75646 │             41.72 │        15303.81 │               39.68 │
│ carriageway_mapping │ certain                                      │     14765 │              8.14 │         3100.54 │                8.04 │

## 12. Result: `bicycle_route_m_ways_distinct_classified`

`bicycle_route_m_ways_distinct_classified` contains one row per way in `bicycle_route_m_ways_distinct`, enriched with all columns produced by Steps 1–9. It carries the original way geometry and length columns unchanged.

The cell below verifies that the row count matches the input exactly. Per thesis §3.4.1 ("the table retains one row per way throughout, 181,318 in and 181,318 out"), the grain must not have changed; any discrepancy would invalidate both downstream pipelines.

**Downstream use.** Per the pipeline diagram, `bicycle_route_m_ways_distinct_classified` feeds:

- `05_bicycle_route_infrastructure_per_side`: unnests to one row per way per side and assigns the UNECE class / evidence basis / classifiability columns (the per-side classification framework from thesis §3.4.2 to §3.4.4).
- Stage 06 spatial join: produces `bicycle_route_m_ways_distinct_classified_per_spatial_unit` and `bicycle_route_infrastructure_per_side_per_spatial_unit`.
- Stage 07 metrics: feeds `bicycle_route_extent_metrics` and `bicycle_route_infrastructure_per_side_metrics`.

In [28]:
# Verify row count matches input exactly — grain must not have changed
n_input      = duckdb.sql("SELECT COUNT(*) FROM bicycle_route_m_ways_distinct").fetchone()[0]
n_classified = duckdb.sql("SELECT COUNT(*) FROM bicycle_route_m_ways_distinct_classified").fetchone()[0]
 
print(f"Input ways:      {n_input:>10,}")
print(f"Classified ways: {n_classified:>10,}")
print(f"Match:           {n_input == n_classified}")

Input ways:         181,318
Classified ways:    181,318
Match:           True


In [29]:
duckdb.sql("""
SELECT DISTINCT mapping_style, infrastructure_confidence
FROM bicycle_route_m_ways_distinct_classified
WHERE mapping_style='carriageway_mapping'
OR  mapping_style='n/a'
""").show(max_width=150)

┌─────────────────────┬──────────────────────────────────────────────┐
│    mapping_style    │          infrastructure_confidence           │
│       varchar       │                   varchar                    │
├─────────────────────┼──────────────────────────────────────────────┤
│ carriageway_mapping │ medium_side_null_unexplained                 │
│ n/a                 │ high_use_sidepath_confirms_no_infrastructure │
│ carriageway_mapping │ high_explained_by_roundabout                 │
│ carriageway_mapping │ high_explained_by_sidepath                   │
│ carriageway_mapping │ certain                                      │
│ n/a                 │ low_no_cycleway_infrastructure_signal        │
└─────────────────────┴──────────────────────────────────────────────┘



The row counts must match exactly. Any discrepancy indicates that a step dropped or duplicated rows, which would invalidate both downstream pipelines.

### Export

Persist `bicycle_route_m_ways_distinct_classified` to disk so the two downstream pipelines (per-side infrastructure and spatial-join-then-metrics) can read it without re-running the full tag-resolution chain. `safe_write_parquet` writes to a `.tmp` path first and then atomically renames, so a concurrent reader either sees the previous snapshot or the new one but never a partial file.

In [31]:
from pathlib import Path
import os

Path("/local/data/vbo226/cache").mkdir(parents=True, exist_ok=True)


def safe_write_parquet(df, final_path):
    tmp_path = final_path + ".tmp"

    df.write_parquet(tmp_path)
    os.replace(tmp_path, final_path)

safe_write_parquet(
    bicycle_route_m_ways_distinct_classified,
    "/local/data/vbo226/cache/bicycle_route_m_ways_distinct_classified.parquet"
)
